# 06｜拆开 State：集合注意力、Energy 距离与残差

对应课程：[L2-01 State 模型深潜](../docs/lessons/L2-01-State模型深潜.md)。

**目标：** 用纯 NumPy 重建 State 的三个关键机制，让你能不看论文解释「为什么损失不能换成 MSE」。

| 单元 | 内容 | 你会拿到什么 |
|---|---|---|
一 | 把细胞当 token：手写集合自注意力 | 置换等变性的实测验证，以及因果掩码为什么不行 |
二 | Energy 距离 vs MSE | MSE 依赖行配对、会因此偏好塌缩解的对照实验 |
三 | 残差加回（predict_residual） | 「学 delta」与「学全量」的收敛对照 |
四 | 参数量计算器 | 两套配置的参数量分解，改维度即可重算 |
五 | 把结论变成判据 | 可复用的检查表 + 诊断题 |

**先记住三条边界：**

1. 本笔记**不是**官方 State 实现，也不加载任何权重。它是按源码结构写的**数学教学模型**，用来验证机制，不能用来评估性能。
2. 本笔记**不安装也不使用** `arc-state` 或 PyTorch（见 `notebook/README.md`）。
3. 单元三的收敛对照是**线性回归 toy**，不是 8 层 Transformer 的训练动力学。它只说明「起点更接近目标」这一件事。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent

SEED = 20260917
rng = np.random.default_rng(SEED)

print("ROOT =", ROOT)
print("numpy =", np.__version__)

## 单元一：一个细胞一个 token

State 的 Transformer 里，序列的每个位置是一个**细胞**，不是一个基因、也不是一个词。这带来一个硬约束：

> **细胞集合没有顺序。** 打乱一个集合里的细胞顺序，模型输出的**集合**必须不变。

源码为此做了三件事（见课文 §3.2）：关掉因果掩码、关掉 RoPE、把 token 词表置零并冻结。下面验证前两件事的区别。

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def set_attention(X, Wq, Wk, Wv, Wo, causal=False):
    """X: (S, d)，每行一个细胞。causal=True 时加下三角因果掩码。"""
    S = X.shape[0]
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    scores = (Q @ K.T) / np.sqrt(Q.shape[-1])
    if causal:
        scores = scores + np.triu(np.ones((S, S)), k=1) * (-1e9)
    A = softmax(scores)
    return (A @ V) @ Wo


S, d, dh = 6, 8, 12        # 6 个细胞，输入 8 维，注意力宽度 12（= heads x head_dim）
scale = 0.3
Wq, Wk, Wv = (rng.normal(scale=scale, size=(d, dh)) for _ in range(3))
Wo = rng.normal(scale=scale, size=(dh, d))
X = rng.normal(size=(S, d))
perm = rng.permutation(S)

Y = set_attention(X, Wq, Wk, Wv, Wo)
Y_perm = set_attention(X[perm], Wq, Wk, Wv, Wo)

Yc = set_attention(X, Wq, Wk, Wv, Wo, causal=True)
Yc_perm = set_attention(X[perm], Wq, Wk, Wv, Wo, causal=True)

d_bi = float(np.abs(Y[perm] - Y_perm).max())
d_causal = float(np.abs(Yc[perm] - Yc_perm).max())

print("双向注意力：||Y[perm] - Y_perm||_max =", f"{d_bi:.2e}")
print("因果掩码  ：||Y[perm] - Y_perm||_max =", f"{d_causal:.2e}")
print()
print("双向：输出跟着输入一起被同样地打乱 -> 置换等变（集合语义成立）")
print("因果：同样的输入换个顺序，输出就变了 -> 它把细胞当成了序列")
assert d_bi < 1e-12 and d_causal > 1e-6

**为什么这件事值一整个单元？** 因为「置换等变」是 §4 那个「不配对」赌注在架构层的落地。如果模型对顺序敏感，那么训练时就必须固定一个顺序——可是 Perturb-seq 的数据里根本没有这个顺序可固定。

第三件事（词表置零冻结）在纯 NumPy 里没法演示，但它是同一条逻辑：**模型接收的是 `inputs_embeds`，不是 token id**，说明作者真的只是借 Transformer 的注意力机制，不是在做语言建模。你可以用这条命令自己确认：

```bash
grep -n "embed_tokens.weight.zero_\|wte.weight.zero_\|use_rotary_embeddings" \
  references/state/src/state/tx/models/utils.py
```

## 单元二：Energy 距离 vs MSE

课文 §4.1 给出闭式：

$$D^2(P, Q) = 2\,\mathbb{E}\|p-q\| - \mathbb{E}\|p-p'\| - \mathbb{E}\|q-q'\|$$

中间那一项是**负号**，它奖励「预测集合自己散得开」。下面用两种候选预测做一个对照实验：

- `P1`：分布正确的预测（逐行接近真值，但**行序是任意的**——这就是真实情况）
- `P2`：塌缩到条件均值的预测（每个细胞都一样）

对每个 trial 随机打乱 `P1` 的行序，然后看两种损失各自选谁。

In [ ]:
def mean_pairwise(A, B):
    """E||a - b||，对 A x B 的所有配对取平均。"""
    D = np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(-1))
    return float(D.mean())


def energy_distance(P, Q):
    return 2 * mean_pairwise(P, Q) - mean_pairwise(P, P) - mean_pairwise(Q, Q)


def mse_paired(P, Q):
    """按行配对的 MSE。注意它假设第 i 行预测对应第 i 行真值。"""
    return float(((P - Q) ** 2).mean())


rng2 = np.random.default_rng(SEED + 1)
G, S2, N_TRIAL = 64, 32, 60

rows = []
for _ in range(N_TRIAL):
    Q = rng2.normal(size=(S2, G)) + rng2.normal(size=G) * 2.0   # 真实扰动细胞集合
    P1 = Q + rng2.normal(scale=0.3, size=Q.shape)               # 分布正确的预测
    P2 = np.repeat(Q.mean(axis=0, keepdims=True), S2, axis=0)   # 塌缩到均值
    perm = rng2.permutation(S2)                                 # 行序任意
    rows.append({
        "mse_correct": mse_paired(P1[perm], Q),
        "mse_collapsed": mse_paired(P2, Q),
        "ed_correct": energy_distance(P1, Q),
        "ed_collapsed": energy_distance(P2, Q),
    })

import pandas as pd
df = pd.DataFrame(rows)
mse_picks_collapsed = float((df.mse_correct > df.mse_collapsed).mean())
ed_picks_collapsed = float((df.ed_correct > df.ed_collapsed).mean())

print(f"{N_TRIAL} 次重复（G={G}, 每集合 {S2} 个细胞）\n")
print(df.mean().round(4).to_string())
print()
print(f"MSE    选中【塌缩解】的比例: {mse_picks_collapsed:.0%}")
print(f"Energy 选中【塌缩解】的比例: {ed_picks_collapsed:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.bar(["MSE"], [df.mse_correct.mean()], color="#4C72B0", label="P1 分布正确（行序任意）")
ax.bar(["MSE"], [df.mse_collapsed.mean()], bottom=[0], width=0.4, color="#DD8452", label="P2 塌缩到均值")
ax.set_title("MSE：越小越好\n（塌缩解更低 -> 它被选中）")
ax.legend(fontsize=8)
ax.set_ylabel("损失")

ax = axes[1]
ax.bar(["Energy"], [df.ed_correct.mean()], color="#4C72B0", label="P1 分布正确")
ax.bar(["Energy"], [df.ed_collapsed.mean()], width=0.4, color="#DD8452", label="P2 塌缩到均值")
ax.set_title("Energy 距离：越小越好\n（正确解更低 -> 它被选中）")
ax.legend(fontsize=8)
ax.set_ylabel("距离")

for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle("同一个预测、同一个真值，两种损失给出相反的排序", fontsize=12)
fig.tight_layout()
plt.show()

**怎么读这张图。** 左图里蓝条（分布正确的预测）比橙条（塌缩解）**高**，所以 MSE 会选橙条——它会把 400 个细胞压成一个均值。右图相反。

差别来自 energy 距离的中间一项：`P2` 把 `E||p-p'||` 做成 0，丢掉了本该得到的离散度奖励，于是被罚。**这就是「不能换 MSE」的机制层面原因**，而不只是「论文这么写」。

再强调一次因果：MSE 偏好塌缩解，不是因为它算错了，而是因为它依赖了一个**数据里不存在的前提**（行对应）。真实 Perturb-seq 里，你拿到的真值细胞和你的预测细胞之间没有天然配对。

## 单元三：残差加回（predict_residual）

课文 §3.4：`predict_residual=true` 让模型只学「变化量」。直觉是——细胞有近两万个基因的值，扰动通常只改变其中一小部分，所以「从对照出发再改一点」比「从零重建一整个表达谱」起点近得多。

下面用**线性回归 toy** 验证这一件事。注意它的边界：这不是 8 层 Transformer 的训练动力学，它只说明起点差异。

In [ ]:
def train_linear(X, Y, steps=400, lr=0.05):
    """最小二乘的梯度下降。返回损失曲线。"""
    W = np.zeros((X.shape[1], Y.shape[1]))
    hist = []
    for _ in range(steps):
        pred = X @ W
        W -= lr * (X.T @ (pred - Y)) / len(X)
        hist.append(float(((pred - Y) ** 2).mean()))
    return W, hist


rng3 = np.random.default_rng(SEED + 2)
n, G3 = 256, 40
X0 = rng3.normal(size=(n, G3)) * 3.0          # 对照表达：尺度大，扰动前后几乎不变
delta = rng3.normal(size=(1, G3)) * 0.05      # 真实效应：比对照小两个数量级
Y0 = X0 + delta

W_full, h_full = train_linear(X0, Y0)                 # 直接学完整映射（predict_residual=false）

def train_residual_curve(X, Y, steps=400, lr=0.05):
    W = np.zeros((X.shape[1], Y.shape[1]))
    hist = []
    for _ in range(steps):
        pred = X + X @ W                              # 关键：预测 = 对照 + 学到的 delta
        W -= lr * (X.T @ (pred - Y)) / len(X)
        hist.append(float(((pred - Y) ** 2).mean()))
    return W, hist

_, h_res = train_residual_curve(X0, Y0)

tol = 5e-3

def first_under(h, tol):
    for i, v in enumerate(h):
        if v < tol:
            return i
    return None

print(f"起始损失  学全量: {h_full[0]:.4f}   学 delta: {h_res[0]:.4f}")
print(f"最终损失  学全量: {h_full[-1]:.2e}   学 delta: {h_res[-1]:.2e}")
print(f"降到 {tol} 所需步数  学全量: {first_under(h_full, tol)}   学 delta: {first_under(h_res, tol)}")
print()
print("两条路径的最终损失相同（都是这个 toy 的最小二乘残差），")
print("差别只在起点和到达速度 —— 这正是 predict_residual 想买的东西。")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h_full, label="predict_residual=false（学完整映射）", color="#DD8452")
ax.plot(h_res, label="predict_residual=true（学 delta）", color="#4C72B0")
ax.set_yscale("log")
ax.set_xlabel("梯度步")
ax.set_ylabel("MSE")
ax.set_title("效应幅度远小于对照表达时，两条路径的起点差两个数量级")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

**边界要写清：** 这个 toy 只说明「起点更接近目标」。真实模型里残差的价值还有第二层——学习目标从「重建 18,533 个数」变成「预测其中变化的那部分」，这会影响梯度的信噪比，进而影响 **需要多少数据才能学到东西**。这一层 toy 演示不了。

**还有一层风险，与 [L1-02](../docs/lessons/L1-02-证据基线与噪声地板.md) 直接相关：** 如果真实 delta 的幅度小于噪声地板，那么残差分支学到的就是噪声。它的表现是训练损失稳定下降但指标不动——详见课文诊断题三。

## 单元四：参数量计算器

课文 §3.6 那张表的计算器。改 `G / d_h / 层数 / heads / head_dim` 就能重算，用来回答「换基因轴到底要重训多少参数」。

**这是手算值，不是实例化实测。** 请以真实模型的 `sum(p.numel() for p in model.parameters() if p.requires_grad)` 为准。

In [ ]:
def count_params(G, d_h, n_layers, n_heads, head_dim, intermediate,
                 D_pert, D_batch=None, output_space="gene", vocab=32000):
    """按 State 的矩阵形状手算参数量。两套编码/解码层按 n_layers=1（单层 Linear）计。"""
    aw = n_heads * head_dim                      # 注意力内部宽度
    attn = (d_h * aw + aw) * 3 + (aw * d_h + d_h)  # q / k / v / o
    mlp = (d_h * intermediate + intermediate) * 2 + (intermediate * d_h + d_h)
    per_layer = attn + mlp + 2 * d_h
    backbone = n_layers * per_layer + d_h

    enc = (D_pert * d_h + d_h) + (G * d_h + d_h)   # pert_encoder + basal_encoder
    if D_batch:
        enc += D_batch * d_h
    proj_out = d_h * G + G

    fdtu = 0
    if output_space == "all":
        h = G // 8
        fdtu = (G * h + h) + (h * G + G)           # final_down_then_up

    frozen = vocab * d_h                           # embed_tokens：分配但置零并冻结
    return {
        "注意力内部宽度": aw,
        "主干 8 层": backbone,
        "三个编码器": enc,
        "project_out": proj_out,
        "final_down_then_up": fdtu,
        "可训练合计": backbone + enc + proj_out + fdtu,
        "冻结的 embed_tokens": frozen,
    }


cfgs = {
    "328 / G=2000（已发布 HVG 权重）": dict(G=2000, d_h=328, n_layers=8, n_heads=12,
                                        head_dim=64, intermediate=3072,
                                        D_pert=2024, D_batch=56, output_space="gene"),
    "768 / G=18533（源码默认，全基因）": dict(G=18533, d_h=768, n_layers=8, n_heads=12,
                                        head_dim=64, intermediate=3072,
                                        D_pert=2024, D_batch=None, output_space="all"),
}

tbl = pd.DataFrame({name: count_params(**cfg) for name, cfg in cfgs.items()})
print("单位：万参数\n")
print((tbl / 1e4).round(1).to_string())

**最容易看走眼的一项：** 768 那套里 `final_down_then_up`（约 8,587 万）比整个主干还大。它只是 `G -> G/8 -> G` 两个矩阵，但因为两头都是 `G=18,533`，参数量随 `G^2` 走。

**这条直接推出一个工程结论：** 把基因轴从 2,000 换到 18,533，真正贵的不是主干（主干可迁移），而是两端那两个随 `G^2` 增长、且**必须从头训练**的模块。

## 单元五：把结论变成判据

跑完请把这张表填进你的实验记录。它是 [L2-01](../docs/lessons/L2-01-State模型深潜.md) §8 的那张表。

In [ ]:
checklist = [
    ("我打算复用哪一套配置（328 还是 768）", "课文 §3.5，两套不能混用"),
    ("该配置下 G_in / G_out / S / d_h 分别是多少", "课文 §2.2"),
    ("换基因轴时哪两层必须重训", "课文 §7 第 1 项；用单元四重算"),
    ("我的靶点表示维度，是否会撞上 2024", "课文 §7 第 4 项（同维异义是大坑）"),
    ("若训练损失下降但预测塌缩成均值，我怀疑哪一项", "课文 §4.1 后两项 / 本笔记单元二"),
]
for i, (q, ref) in enumerate(checklist, 1):
    print(f"{i}. {q}\n   依据 -> {ref}\n")

print("=" * 60)
print("诊断题：把 energy 换成 MSE 后，训练损失更低、但 400 个细胞几乎相同。")
print("  (a) 为什么 MSE 会更低？        -> MSE 的最小值在条件期望处")
print("  (b) 为什么本赛指标会更差？      -> README §5 群体分布线；DEG 线也会受损")
print("  (c) energy 的哪一项在起作用？   -> -E||p-p'|| - E||q-q'||（离散度奖励）")
print("完整答案见课文 §9.1 的折叠块。")